<a href="https://colab.research.google.com/github/albert-magarire/Data-Science-and-ML/blob/main/HoverNet_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import subprocess, sys, os

for pkg in ['imgaug==0.4.0', 'tensorboardX', 'docopt', 'termcolor',
            'scikit-image', 'scikit-learn', 'scipy', 'tqdm',
            'opencv-python-headless', 'gdown']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])

REPO_PATH = '/content/hover_net'
if not os.path.isdir(REPO_PATH):
    os.system(f'git clone https://github.com/vqdang/hover_net.git {REPO_PATH}')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
import os

REPO_PATH = '/content/hover_net'

# --- DATA PATHS (update to match your Google Drive) ---
GDRIVE_TRAIN_PATH = '/content/drive/MyDrive/MoNuSegSplit_80_20/train'
GDRIVE_VALID_PATH = '/content/drive/MyDrive/MoNuSegSplit_80_20/val'
GDRIVE_TEST_PATH  = '/content/drive/MyDrive/MoNuSegSplit_80_20/test'
IMG_SUBDIR  = 'images'
MASK_SUBDIR = 'masks'

PATCH_ROOT      = '/content/hovernet_patches'
TRAIN_PATCH_DIR = os.path.join(PATCH_ROOT, 'train')
VALID_PATCH_DIR = os.path.join(PATCH_ROOT, 'valid')
TEST_PATCH_DIR  = os.path.join(PATCH_ROOT, 'test')

# --- MODEL ---
MODEL_MODE           = 'fast'
TYPE_CLASSIFICATION  = False
NR_TYPES             = None

# ImageNet-pretrained Preact-ResNet50 backbone (downloaded in next cell)
PRETRAINED_PATH = '/content/pretrained/pretrained_net.tar'

# --- TRAINING ---
NR_EPOCHS_PHASE1  = 50
NR_EPOCHS_PHASE2  = 50
BATCH_SIZE_PHASE1 = 8
BATCH_SIZE_PHASE2 = 4
LEARNING_RATE     = 1.0e-4
NR_DATA_WORKERS   = 4
LOG_DIR           = '/content/hovernet_logs'
GPU_IDS           = '0'
SEED              = 10

print('Configuration loaded.')
print(f'  Pretrained:   {PRETRAINED_PATH}')
print(f'  Total epochs: {NR_EPOCHS_PHASE1 + NR_EPOCHS_PHASE2}')

Configuration loaded.
  Pretrained:   /content/pretrained/pretrained_net.tar
  Total epochs: 100


In [ ]:
import os, gdown

pretrained_dir = '/content/pretrained'
os.makedirs(pretrained_dir, exist_ok=True)

if not os.path.exists(PRETRAINED_PATH):
    print('Downloading ImageNet-pretrained Preact-ResNet50 backbone...')
    gdown.download(
        'https://drive.google.com/uc?id=1KntZge40tAHgyXmHYVqZZ5d2p_4Qr2l5',
        PRETRAINED_PATH, quiet=False
    )
    print(f'Saved to {PRETRAINED_PATH}')
else:
    print(f'Pretrained weights already exist at {PRETRAINED_PATH}')

assert os.path.exists(PRETRAINED_PATH), (
    f'Failed to download pretrained weights. '
    f'Manually download from https://drive.google.com/uc?id=1KntZge40tAHgyXmHYVqZZ5d2p_4Qr2l5 '
    f'and place at {PRETRAINED_PATH}'
)

Downloading...
From: https://drive.google.com/uc?id=1KntZge40tAHgyXmHYVqZZ5d2p_4Qr2l5
To: /content/pretrained/pretrained_net.tar

  0%|          | 0.00/94.2M [00:00<?, ?B/s]
  1%|          | 1.05M/94.2M [00:00<00:08, 10.4MB/s]
  8%|▊         | 7.86M/94.2M [00:00<00:01, 43.8MB/s]
 27%|██▋       | 25.2M/94.2M [00:00<00:00, 102MB/s] 
 38%|███▊      | 35.7M/94.2M [00:00<00:00, 97.3MB/s]
 53%|█████▎    | 50.3M/94.2M [00:00<00:00, 114MB/s] 
 66%|██████▌   | 61.9M/94.2M [00:00<00:00, 103MB/s]
 79%|███████▉  | 74.4M/94.2M [00:00<00:00, 109MB/s]
100%|██████████| 94.2M/94.2M [00:00<00:00, 106MB/s]

Saved to /content/pretrained/pretrained_net.tar


In [ ]:
import os, sys, importlib, numpy as np

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

# --- Patch np.lib.pad → np.pad ---
pe_path = os.path.join(REPO_PATH, 'misc/patch_extractor.py')
if os.path.exists(pe_path):
    with open(pe_path, 'r') as f:
        code = f.read()
    if 'np.lib.pad' in code:
        with open(pe_path, 'w') as f:
            f.write(code.replace('np.lib.pad', 'np.pad'))
        print('Patched: np.lib.pad -> np.pad')

# --- Patch np.sctypes for imgaug ---
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'float': [np.float16, np.float32, np.float64],
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'complex': [np.complex64, np.complex128]
    }
    if hasattr(np, 'float128'):
        np.sctypes['float'].append(np.float128)
    print('Patched: np.sctypes for imgaug')

# --- Patch viz_step_output shape alignment ---
rd_path = os.path.join(REPO_PATH, 'models/hovernet/run_desc.py')
if os.path.exists(rd_path):
    with open(rd_path, 'r') as f:
        code = f.read()
    old_line = "aligned_shape = np.min(np.array(aligned_shape), axis=0)[1:3]"
    if old_line in code:
        new_logic = (
            "    shapes = [imgs.shape, true_np.shape, pred_np.shape]\n"
            "    min_h = min(s[1] for s in shapes)\n"
            "    min_w = min(s[2] for s in shapes)\n"
            "    aligned_shape = [min_h, min_w]"
        )
        code = code.replace(
            "aligned_shape = [list(imgs.shape), list(true_np.shape), list(pred_np.shape)]",
            "# aligned_shape = ..."
        )
        code = code.replace(old_line, new_logic)
        with open(rd_path, 'w') as f:
            f.write(code)
        print('Patched: viz_step_output shape alignment')

# --- Inject focal_loss into run_desc.py ---
with open(rd_path, 'r') as f:
    code = f.read()

focal_loss_code = '''
def focal_loss(true, pred):
    gamma = 2.0
    eps = 1e-7
    pred = torch.clamp(pred, eps, 1. - eps)
    pt = (true * pred).sum(dim=-1)
    loss = -((1 - pt) ** gamma) * torch.log(pt)
    return loss.mean()

'''

if 'def focal_loss' not in code:
    code = code.replace('def train_step(', focal_loss_code + 'def train_step(')
    print('Injected: focal_loss function')

if '"focal": focal_loss' not in code:
    code = code.replace('"msge": msge_loss,', '"msge": msge_loss,\n        "focal": focal_loss,')
    print('Registered: focal in loss_func_dict')

with open(rd_path, 'w') as f:
    f.write(code)

# --- Patch JSON serialization (float32) ---
log_path = os.path.join(REPO_PATH, 'run_utils/callbacks/logging.py')
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        code = f.read()
    if 'class NumpyEncoder' not in code:
        encoder = '''
import json as _json
import numpy as _np
class NumpyEncoder(_json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, _np.integer): return int(obj)
        elif isinstance(obj, _np.floating): return float(obj)
        elif isinstance(obj, _np.ndarray): return obj.tolist()
        return super().default(obj)

'''
        code = code.replace('class LoggingEpochOutput', encoder + 'class LoggingEpochOutput')
        code = code.replace('json.dump(json_data, json_file)', 'json.dump(json_data, json_file, cls=NumpyEncoder)')
        with open(log_path, 'w') as f:
            f.write(code)
        print('Patched: JSON NumpyEncoder')

# --- Patch NaN Center of Mass in targets.py ---
tgt_path = os.path.join(REPO_PATH, 'models/hovernet/targets.py')
if os.path.exists(tgt_path):
    with open(tgt_path, 'r') as f:
        code = f.read()
    old_block = "inst_com[0] = int(inst_com[0] + 0.5)"
    if old_block in code and "if np.any(np.isnan(inst_com))" not in code:
        code = code.replace(old_block, "if np.any(np.isnan(inst_com)): continue\n        inst_com[0] = int(inst_com[0] + 0.5)")
        with open(tgt_path, 'w') as f:
            f.write(code)
        print('Patched: NaN center-of-mass guard')

# Force-reload all patched modules
for mod_name in list(sys.modules.keys()):
    if any(k in mod_name for k in ['hover', 'run_utils', 'misc.patch', 'dataloader']):
        del sys.modules[mod_name]

print('All patches applied.')

All patches applied.


In [ ]:
import glob, pathlib, shutil, warnings, cv2, numpy as np
from scipy import ndimage
import tqdm

sys.path.insert(0, REPO_PATH)
from misc.patch_extractor import PatchExtractor

WIN_SIZE  = [540, 540]
STEP_SIZE = [164, 164]
IMG_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.tiff', '.tif', '.bmp']

def load_instance_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if mask is None:
        return None
    if mask.ndim == 3:
        if mask.shape[2] == 4:
            mask = mask[:, :, 0]
        elif mask.shape[2] == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        else:
            mask = mask[:, :, 0]
    mask = mask.astype(np.int32)
    unique_vals = np.unique(mask)
    non_bg = unique_vals[unique_vals != 0]
    if len(non_bg) <= 1:
        binary = (mask > 0).astype(np.uint8)
        labeled, _ = ndimage.label(binary)
        mask = labeled.astype(np.int32)
    return mask

def find_mask_path(img_stem, mask_dir):
    for base in [img_stem, img_stem + '_mask']:
        for ext in IMG_EXTENSIONS:
            for e in [ext, ext.upper()]:
                path = os.path.join(mask_dir, base + e)
                if os.path.exists(path):
                    return path
    return None

def extract_and_save(file_list, out_dir, split_name, img_dir, mask_dir):
    xtractor = PatchExtractor(WIN_SIZE, STEP_SIZE)
    total_patches, skipped = 0, 0
    for img_path in tqdm.tqdm(file_list, desc=f'Extracting {split_name}'):
        stem = pathlib.Path(img_path).stem
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            skipped += 1; continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mask_path = find_mask_path(stem, mask_dir)
        if mask_path is None:
            skipped += 1; continue
        inst_map = load_instance_mask(mask_path)
        if inst_map is None:
            skipped += 1; continue
        if img_rgb.shape[:2] != inst_map.shape[:2]:
            inst_map = cv2.resize(inst_map, (img_rgb.shape[1], img_rgb.shape[0]),
                                  interpolation=cv2.INTER_NEAREST).astype(np.int32)
        stacked = np.concatenate([img_rgb, inst_map[:, :, np.newaxis]], axis=-1)
        patches = xtractor.extract(stacked, 'mirror')
        for idx, patch in enumerate(patches):
            np.save(os.path.join(out_dir, f'{stem}_{idx:04d}.npy'), patch)
        total_patches += len(patches)
    print(f'  {split_name}: {total_patches} patches from {len(file_list)} images ({skipped} skipped)')
    return total_patches

def get_image_files(base_dir):
    img_dir  = os.path.join(base_dir, IMG_SUBDIR)
    mask_dir = os.path.join(base_dir, MASK_SUBDIR)
    assert os.path.isdir(img_dir),  f'Not found: {img_dir}'
    assert os.path.isdir(mask_dir), f'Not found: {mask_dir}'
    files = []
    for ext in IMG_EXTENSIONS:
        files += glob.glob(os.path.join(img_dir, f'*{ext}'))
        files += glob.glob(os.path.join(img_dir, f'*{ext.upper()}'))
    return sorted(set(files)), img_dir, mask_dir

for split_name, src_path, dst_dir in [
    ('train', GDRIVE_TRAIN_PATH, TRAIN_PATCH_DIR),
    ('valid', GDRIVE_VALID_PATH, VALID_PATCH_DIR),
    ('test',  GDRIVE_TEST_PATH,  TEST_PATCH_DIR),
]:
    if os.path.isdir(dst_dir):
        shutil.rmtree(dst_dir)
    os.makedirs(dst_dir)
    files, img_dir, mask_dir = get_image_files(src_path)
    print(f'{split_name}: {len(files)} images found')
    extract_and_save(files, dst_dir, split_name, img_dir, mask_dir)

print('Data preparation complete.')

train: 33 images found



Extracting train: 100%|██████████| 33/33 [00:44<00:00,  1.34s/it]


  train: 1617 patches from 33 images (0 skipped)
valid: 7 images found



Extracting valid: 100%|██████████| 7/7 [00:06<00:00,  1.06it/s]


  valid: 343 patches from 7 images (0 skipped)
test: 11 images found



Extracting test: 100%|██████████| 11/11 [00:15<00:00,  1.41s/it]

  test: 539 patches from 11 images (0 skipped)
Data preparation complete.


In [ ]:
import matplotlib.pyplot as plt, numpy as np, glob, pathlib

sample_files = sorted(glob.glob(os.path.join(TRAIN_PATCH_DIR, '*.npy')))[:4]
assert len(sample_files) > 0, 'No training patches found!'

fig, axes = plt.subplots(len(sample_files), 2, figsize=(10, 4 * len(sample_files)))
if len(sample_files) == 1:
    axes = [axes]
for i, fpath in enumerate(sample_files):
    data = np.load(fpath)
    axes[i][0].imshow(data[..., :3].astype('uint8'))
    axes[i][0].set_title(pathlib.Path(fpath).stem); axes[i][0].axis('off')
    inst = data[..., 3].astype('int32')
    axes[i][1].imshow(inst, cmap='nipy_spectral')
    axes[i][1].set_title(f'Instances: {len(np.unique(inst)) - 1}'); axes[i][1].axis('off')
plt.tight_layout(); plt.show()

for label, d in [('Train', TRAIN_PATCH_DIR), ('Valid', VALID_PATCH_DIR), ('Test', TEST_PATCH_DIR)]:
    n = len(glob.glob(os.path.join(d, '*.npy')))
    print(f'  {label}: {n} patches')

  Train: 1617 patches
  Valid: 343 patches
  Test: 539 patches


In [ ]:
import cv2; cv2.setNumThreads(0)
import json, shutil, random, glob, time
import numpy as np, torch, torch.optim as optim
from torch.nn import DataParallel
from torch.utils.data import DataLoader
from tensorboardX import SummaryWriter

import sys, os, importlib
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

from dataloader.train_loader import FileLoader
from models.hovernet.net_desc import create_model
from models.hovernet.targets import gen_targets, prep_sample
from models.hovernet.run_desc import (
    proc_valid_step_output, train_step, valid_step, viz_step_output
)
from run_utils.engine import RunEngine, Events
from run_utils.callbacks.base import (
    AccumulateRawOutput, BaseCallbacks, PeriodicSaver,
    ProcessAccumulatedRawOutput, ScalarMovingAverage,
    ScheduleLr, TrackLr, VisualizeOutput, TriggerEngine,
)
from run_utils.callbacks.logging import LoggingEpochOutput
from run_utils.utils import check_manual_seed, convert_pytorch_checkpoint
from misc.utils import rm_n_mkdir

def worker_init_fn(worker_id):
    worker_info = torch.utils.data.get_worker_info()
    worker_seed = torch.randint(0, 2**32, (1,))[0].cpu().item() + worker_id
    worker_info.dataset.setup_augmentor(worker_id, worker_seed)

class BestCheckpointSaver(BaseCallbacks):
    def __init__(self, metric_name='valid-np_dice', mode='max', filename='net_best_checkpoint'):
        super().__init__()
        self.metric_name = metric_name
        self.mode = mode
        self.filename = filename
        self.best_value = -float('inf') if mode == 'max' else float('inf')
        self.best_epoch = -1

    def run(self, state, event):
        if not state.logging:
            return
        current_epoch_str = str(state.global_state.curr_epoch if state.global_state else state.curr_epoch)
        try:
            with open(state.log_info['json_file']) as fh:
                json_data = json.load(fh)
        except Exception:
            return
        if current_epoch_str not in json_data:
            return
        epoch_stats = json_data[current_epoch_str]
        if self.metric_name not in epoch_stats:
            return
        current_value = float(epoch_stats[self.metric_name])
        is_best = (
            (self.mode == 'max' and current_value > self.best_value) or
            (self.mode == 'min' and current_value < self.best_value)
        )
        if is_best:
            self.best_value = current_value
            self.best_epoch = int(current_epoch_str)
            print(f'\n  [BEST] Epoch {self.best_epoch}: {self.metric_name} = {current_value:.4f}')
            for net_name, net_info in state.run_info.items():
                checkpoint = {k: v.state_dict() for k, v in net_info.items() if k != 'extra_info'}
                save_path = os.path.join(state.log_dir, f'{self.filename}.tar')
                torch.save(checkpoint, save_path)

def run_phase(phase_info, phase_dir, prev_phase_dir=None, seed=SEED):
    nr_gpus = max(1, torch.cuda.device_count())
    check_manual_seed(seed)
    rm_n_mkdir(phase_dir)
    tfwriter = SummaryWriter(log_dir=phase_dir)
    json_log_file = os.path.join(phase_dir, 'stats.json')
    with open(json_log_file, 'w') as fh:
        json.dump({}, fh)
    log_info = {'json_file': json_log_file, 'tfwriter': tfwriter}

    if MODEL_MODE == 'original':
        act_shape, out_shape = [270, 270], [80, 80]
    else:
        act_shape, out_shape = [256, 256], [164, 164]
    shape_info = {
        'train': {'input_shape': act_shape, 'mask_shape': out_shape},
        'valid': {'input_shape': act_shape, 'mask_shape': out_shape},
    }

    def build_loader(patch_dir, mode, batch_size, nr_procs):
        file_list = sorted(glob.glob(os.path.join(patch_dir, '*.npy')))
        assert len(file_list) > 0, f'No .npy files in {patch_dir}'
        print(f'  {mode:5s}: {len(file_list)} patches')
        dataset = FileLoader(
            file_list, mode=mode, with_type=TYPE_CLASSIFICATION,
            setup_augmentor=(nr_procs == 0),
            target_gen=phase_info['target_info']['gen'],
            **shape_info[mode]
        )
        return DataLoader(
            dataset, num_workers=nr_procs, batch_size=batch_size * nr_gpus,
            shuffle=(mode == 'train'), drop_last=(mode == 'train'),
            worker_init_fn=worker_init_fn,
        )

    print('Loading datasets...')
    loaders = {
        'train': build_loader(TRAIN_PATCH_DIR, 'train', phase_info['batch_size']['train'], NR_DATA_WORKERS),
        'valid': build_loader(VALID_PATCH_DIR, 'valid', phase_info['batch_size']['valid'], max(1, NR_DATA_WORKERS // 2)),
    }

    net_run_info = {}
    for net_name, net_info in phase_info['run_info'].items():
        net = net_info['desc']()
        pretrained = net_info['pretrained']
        if pretrained is not None:
            if pretrained == -1:
                assert prev_phase_dir is not None
                prev_stats_path = os.path.join(prev_phase_dir, 'stats.json')
                with open(prev_stats_path) as fh:
                    prev_stats = json.load(fh)
                last_epoch = max(int(e) for e in prev_stats.keys())
                pretrained = os.path.join(prev_phase_dir, f'{net_name}_epoch={last_epoch}.tar')
                print(f'  Auto-loading Phase 1 checkpoint: {pretrained}')
                state_dict = torch.load(pretrained)['desc']
            else:
                print(f'  Loading pretrained weights: {pretrained}')
                ext = pretrained.rsplit('.', 1)[-1]
                if ext == 'npz':
                    state_dict = {k: torch.from_numpy(v) for k, v in dict(np.load(pretrained)).items()}
                else:
                    state_dict = torch.load(pretrained)['desc']
            state_dict = convert_pytorch_checkpoint(state_dict)
            missing, unexpected = net.load_state_dict(state_dict, strict=False)
            if missing:
                print(f'  Missing keys ({len(missing)}): {missing[:3]}')
            if unexpected:
                print(f'  Unexpected keys ({len(unexpected)}): {unexpected[:3]}')

        net = DataParallel(net).to('cuda')
        opt_cls, opt_kwargs = net_info['optimizer']
        optimizer = opt_cls(net.parameters(), **opt_kwargs)
        scheduler = net_info['lr_scheduler'](optimizer)
        net_run_info[net_name] = {
            'desc': net, 'optimizer': optimizer,
            'lr_scheduler': scheduler, 'extra_info': net_info['extra_info'],
        }

    nr_type = NR_TYPES
    run_engine_opt = {
        'train': {
            'run_step': train_step,
            'callbacks': {
                Events.STEP_COMPLETED: [ScalarMovingAverage()],
                Events.EPOCH_COMPLETED: [
                    TrackLr(), PeriodicSaver(per_n_epoch=5),
                    VisualizeOutput(viz_step_output), LoggingEpochOutput(),
                    TriggerEngine('valid'), ScheduleLr(),
                ],
            },
        },
        'valid': {
            'run_step': valid_step,
            'callbacks': {
                Events.STEP_COMPLETED: [AccumulateRawOutput()],
                Events.EPOCH_COMPLETED: [
                    ProcessAccumulatedRawOutput(
                        lambda a: proc_valid_step_output(a, nr_types=nr_type)
                    ),
                    LoggingEpochOutput(),
                    BestCheckpointSaver('valid-np_dice', 'max'),
                ],
            },
        },
    }

    runner_dict = {
        name: RunEngine(
            dataloader=loaders[name], engine_name=name,
            run_step=opt['run_step'], run_info=net_run_info, log_info=log_info,
        )
        for name, opt in run_engine_opt.items()
    }
    for name, runner in runner_dict.items():
        for event, cbs in run_engine_opt[name]['callbacks'].items():
            for cb in cbs:
                if cb.engine_trigger:
                    cb.triggered_engine = runner_dict[cb.triggered_engine_name]
                runner.add_event_handler(event, cb)
        runner.state.logging = True
        runner.state.log_dir = phase_dir
    runner_dict['train'].run(phase_info['nr_epochs'])

def build_phase_list():
    nr_type = NR_TYPES
    loss_cfg = {
        'np': {'focal': 1, 'dice': 1},
        'hv': {'mse': 1, 'msge': 1},
    }
    if nr_type is not None:
        loss_cfg['tp'] = {'focal': 1, 'dice': 1}
    print(f'Loss config: {loss_cfg}')
    return [
        {
            'run_info': {
                'net': {
                    'desc': lambda: create_model(input_ch=3, nr_types=nr_type, freeze=True, mode=MODEL_MODE),
                    'optimizer': [optim.Adam, {'lr': LEARNING_RATE, 'betas': (0.9, 0.999)}],
                    'lr_scheduler': lambda x: optim.lr_scheduler.StepLR(x, 25),
                    'extra_info': {'loss': loss_cfg},
                    'pretrained': PRETRAINED_PATH,
                },
            },
            'target_info': {'gen': (gen_targets, {}), 'viz': (prep_sample, {})},
            'batch_size': {'train': BATCH_SIZE_PHASE1, 'valid': BATCH_SIZE_PHASE1},
            'nr_epochs': NR_EPOCHS_PHASE1,
        },
        {
            'run_info': {
                'net': {
                    'desc': lambda: create_model(input_ch=3, nr_types=nr_type, freeze=False, mode=MODEL_MODE),
                    'optimizer': [optim.Adam, {'lr': LEARNING_RATE * 0.5, 'betas': (0.9, 0.999)}],
                    'lr_scheduler': lambda x: optim.lr_scheduler.CosineAnnealingLR(x, T_max=NR_EPOCHS_PHASE2, eta_min=1e-6),
                    'extra_info': {'loss': loss_cfg},
                    'pretrained': -1,
                },
            },
            'target_info': {'gen': (gen_targets, {}), 'viz': (prep_sample, {})},
            'batch_size': {'train': BATCH_SIZE_PHASE2, 'valid': BATCH_SIZE_PHASE2},
            'nr_epochs': NR_EPOCHS_PHASE2,
        },
    ]

print('Training infrastructure ready.')

Training infrastructure ready.


In [ ]:
import time, torch
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_IDS
assert torch.cuda.is_available(), 'No GPU! Go to Runtime → Change runtime type → GPU'

phase_list = build_phase_list()
prev_phase_dir = None

for phase_idx, phase_info in enumerate(phase_list):
    phase_dir = os.path.join(LOG_DIR, f'{phase_idx:02d}')
    label = 'FROZEN backbone' if phase_idx == 0 else 'FULL model'
    print(f'\n{"="*60}')
    print(f'  PHASE {phase_idx+1}/{len(phase_list)}: {label} | Epochs: {phase_info["nr_epochs"]}')
    print(f'{"="*60}')
    t0 = time.time()
    run_phase(phase_info, phase_dir, prev_phase_dir=prev_phase_dir)
    print(f'  Phase {phase_idx+1} finished in {(time.time()-t0)/60:.1f} min.')
    prev_phase_dir = phase_dir

print(f'\n{"="*60}')
print('  TRAINING COMPLETE')
print(f'{"="*60}')

Loss config: {'np': {'focal': 1, 'dice': 1}, 'hv': {'mse': 1, 'msge': 1}}

  PHASE 1/2: FROZEN backbone | Epochs: 50
Using manual seed: 10
Loading datasets...
  train: 1617 patches
  valid: 343 patches
  Loading pretrained weights: /content/pretrained/pretrained_net.tar
  Missing keys (279): ['conv_bot.weight', 'decoder.np.u3.conva.weight', 'decoder.np.u3.dense.units.0.preact_bna/bn.weight']
----------------EPOCH 1



Processing: |          | 0/202[00:00<?,?it/s]Batch = nan|EMA = nan
Processing: |          | 1/202[00:00<01:01, 3.25it/s]Batch = 224.89156|EMA = 224.89156
Processing: |          | 2/202[00:00<00:38, 5.15it/s]Batch = 229.83974|EMA = 225.13896
Processing: |1         | 3/202[00:00<00:31, 6.40it/s]Batch = 178.49347|EMA = 222.80669
Processing: |1         | 4/202[00:00<00:27, 7.23it/s]Batch = 177.15800|EMA = 220.52426
Processing: |2         | 5/202[00:00<00:25, 7.82it/s]Batch = 143.87854|EMA = 216.69197
Processing: |2         | 6/202[00:00<00:23, 8.21it/s]Batch = 136.75653|EMA = 212.69520
Processing: |3         | 7/202[00:00<00:22, 8.49it/s]Batch = 131.99292|EMA = 208.66008
Processing: |3         | 8/202[00:01<00:22, 8.69it/s]Batch = 103.62843|EMA = 203.40850
Processing: |4         | 9/202[00:01<00:21, 8.82it/s]Batch = 90.43068|EMA = 197.75961 
Processing: |4         | 10/202[00:01<00:21, 8.91it/s]Batch = 87.80222|EMA = 192.26174
Processing: |5         | 11/202[00:01<00:21, 8.98it/s]Batch = 

------train-loss_np_focal : 0.23620
------train-loss_np_dice  : 0.50713
------train-loss_hv_mse   : 0.37481
------train-loss_hv_msge  : 1.84963
------train-overall_loss  : 2.96777
------train-lr-net        : 0.00010



Processing: |          | 0/43[00:00<?,?it/s]
Processing: |2         | 1/43[00:00<00:09, 4.66it/s]
Processing: |6         | 3/43[00:00<00:04, 9.41it/s]
Processing: |#1        | 5/43[00:00<00:03,11.53it/s]
Processing: |#6        | 7/43[00:00<00:02,12.75it/s]
Processing: |##        | 9/43[00:00<00:02,13.43it/s]
Processing: |##5       | 11/43[00:00<00:02,13.85it/s]
Processing: |###       | 13/43[00:01<00:02,14.23it/s]
Processing: |###4      | 15/43[00:01<00:01,14.45it/s]
Processing: |###9      | 17/43[00:01<00:01,14.54it/s]
Processing: |####4     | 19/43[00:01<00:01,14.65it/s]
Processing: |####8     | 21/43[00:01<00:01,14.71it/s]
Processing: |#####3    | 23/43[00:01<00:01,14.76it/s]
Processing: |#####8    | 25/43[00:01<00:01,15.02it/s]
Processing: |######2   | 27/43[00:01<00:01,14.93it/s]
Processing: |######7   | 29/43[00:02<00:00,14.90it/s]
Processing: |#######2  | 31/43[00:02<00:00,14.90it/s]
Processing: |#######6  | 33/43[00:02<00:00,14.87it/s]
Processing: |########1 | 35/43[00:02<00:0

------valid-np_acc  : 0.82647
------valid-np_dice : 0.66742
------valid-hv_mse  : 0.63922

  [BEST] Epoch 1: valid-np_dice = 0.6674
----------------EPOCH 2


...

------valid-np_acc  : 0.91370
------valid-np_dice : 0.82090
------valid-hv_mse  : 0.06640
----------------EPOCH 50



Processing: |          | 0/404[00:00<?,?it/s]Batch = nan|EMA = nan
Processing: |          | 1/404[00:00<02:28, 2.70it/s]Batch = 0.68080|EMA = 0.57460
Processing: |          | 3/404[00:00<01:07, 5.96it/s]Batch = 0.48919|EMA = 0.56903
Processing: |1         | 5/404[00:00<00:52, 7.61it/s]Batch = 0.45302|EMA = 0.56153
Processing: |1         | 7/404[00:00<00:46, 8.58it/s]Batch = 0.45305|EMA = 0.55156
Processing: |2         | 9/404[00:01<00:43, 9.17it/s]Batch = 0.59467|EMA = 0.55308
Processing: |2         | 11/404[00:01<00:41, 9.54it/s]Batch = 0.47524|EMA = 0.55009
Processing: |3         | 13/404[00:01<00:40, 9.77it/s]Batch = 0.45203|EMA = 0.54296
Processing: |3         | 15/404[00:01<00:39, 9.94it/s]Batch = 0.62931|EMA = 0.54872
Processing: |4         | 17/404[00:01<00:38,10.05it/s]Batch = 0.66736|EMA = 0.55295
Processing: |4         | 19/404[00:02<00:38,10.12it/s]Batch = 0.58183|EMA = 0.56483
Processing: |5         | 21/404[00:02<00:37,10.17it/s]Batch = 0.78874|EMA = 0.57271
Processing: |

------train-loss_np_focal : 0.08026
------train-loss_np_dice  : 0.21100
------train-loss_hv_mse   : 0.02884
------train-loss_hv_msge  : 0.23548
------train-overall_loss  : 0.55558
------train-lr-net        : 0.00000



Processing: |          | 0/86[00:00<?,?it/s]
Processing: |1         | 1/86[00:00<00:09, 9.16it/s]
Processing: |5         | 5/86[00:00<00:03,22.45it/s]
Processing: |9         | 8/86[00:00<00:03,25.29it/s]
Processing: |#2        | 11/86[00:00<00:02,26.80it/s]
Processing: |#6        | 14/86[00:00<00:02,27.69it/s]
Processing: |#9        | 17/86[00:00<00:02,28.28it/s]
Processing: |##4       | 21/86[00:00<00:02,29.06it/s]
Processing: |##7       | 24/86[00:00<00:02,29.20it/s]
Processing: |###1      | 27/86[00:00<00:02,29.28it/s]
Processing: |###4      | 30/86[00:01<00:01,29.08it/s]
Processing: |###8      | 33/86[00:01<00:01,28.97it/s]
Processing: |####1     | 36/86[00:01<00:01,29.16it/s]
Processing: |####5     | 39/86[00:01<00:01,29.07it/s]
Processing: |####8     | 42/86[00:01<00:01,29.10it/s]
Processing: |#####2    | 45/86[00:01<00:01,29.01it/s]
Processing: |#####5    | 48/86[00:01<00:01,28.96it/s]
Processing: |#####9    | 51/86[00:01<00:01,29.09it/s]
Processing: |######2   | 54/86[00:01<00

------valid-np_acc  : 0.91427
------valid-np_dice : 0.82184
------valid-hv_mse  : 0.06675
  Phase 2 finished in 35.9 min.

  TRAINING COMPLETE


In [ ]:
import json, matplotlib.pyplot as plt

print('=' * 60)
print('  TRAINING RESULTS')
print('=' * 60)

for phase_idx in range(2):
    phase_dir = os.path.join(LOG_DIR, f'{phase_idx:02d}')
    stats_path = os.path.join(phase_dir, 'stats.json')
    if not os.path.exists(stats_path):
        continue
    with open(stats_path) as fh:
        stats = json.load(fh)
    epochs = sorted(int(e) for e in stats.keys())
    if not epochs:
        continue
    train_loss = [stats[str(e)].get('train-overall_loss', float('nan')) for e in epochs]
    valid_dice = [stats[str(e)].get('valid-np_dice', float('nan')) for e in epochs]
    valid_acc  = [stats[str(e)].get('valid-np_acc', float('nan')) for e in epochs]
    best_idx = max(range(len(valid_dice)), key=lambda i: valid_dice[i] if valid_dice[i] == valid_dice[i] else -1)
    print(f'\n  Phase {phase_idx+1}: Best dice = {valid_dice[best_idx]:.4f} (epoch {epochs[best_idx]})')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_loss, 'b-o', markersize=2); axes[0].set_title(f'Phase {phase_idx+1} Loss'); axes[0].grid(True, alpha=0.3)
    axes[1].plot(epochs, valid_dice, 'g-o', markersize=2, label='Dice')
    axes[1].plot(epochs, valid_acc, 'r-s', markersize=2, label='Acc')
    axes[1].axvline(epochs[best_idx], color='k', ls='--', alpha=0.5)
    axes[1].set_title(f'Phase {phase_idx+1} Validation'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

  TRAINING RESULTS

  Phase 1: Best dice = 0.7930 (epoch 35)

  Phase 2: Best dice = 0.8263 (epoch 26)


In [ ]:
import shutil

GDRIVE_SAVE_DIR = '/content/drive/MyDrive/hovernet_checkpoints'
os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)

for phase_idx in range(2):
    src = os.path.join(LOG_DIR, f'{phase_idx:02d}', 'net_best_checkpoint.tar')
    dst = os.path.join(GDRIVE_SAVE_DIR, f'phase{phase_idx+1}_best.tar')
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied: {src} → {dst}')

if os.path.isdir(LOG_DIR):
    log_dst = os.path.join(GDRIVE_SAVE_DIR, 'logs')
    if os.path.isdir(log_dst):
        shutil.rmtree(log_dst)
    shutil.copytree(LOG_DIR, log_dst)
    print(f'Logs saved to: {log_dst}')

Copied: /content/hovernet_logs/00/net_best_checkpoint.tar → /content/drive/MyDrive/hovernet_checkpoints/phase1_best.tar
Copied: /content/hovernet_logs/01/net_best_checkpoint.tar → /content/drive/MyDrive/hovernet_checkpoints/phase2_best.tar
Logs saved to: /content/drive/MyDrive/hovernet_checkpoints/logs


In [ ]:
import numpy as np, torch, torch.nn.functional as F, glob, cv2, tqdm, shutil, os
from torch.utils.data import DataLoader
import importlib, matplotlib.pyplot as plt

import dataloader.train_loader
importlib.reload(dataloader.train_loader)
from dataloader.train_loader import FileLoader
from models.hovernet.net_desc import create_model
from models.hovernet.targets import gen_targets

PRED_SAVE_DIR = '/content/hovernet_predictions/test'

def calculate_metrics(gt_mask, pred_mask):
    gt = (gt_mask > 0).astype(np.uint8).flatten()
    pr = (pred_mask > 0).astype(np.uint8).flatten()
    TP = np.sum(pr * gt)
    TN = np.sum((1 - pr) * (1 - gt))
    FP = np.sum(pr * (1 - gt))
    FN = np.sum((1 - pr) * gt)
    safe = lambda n, d: n / d if d > 0 else 0.0
    return {
        'Accuracy':  safe(TP + TN, TP + TN + FP + FN),
        'Precision': safe(TP, TP + FP),
        'Recall':    safe(TP, TP + FN),
        'Dice':      safe(2 * TP, 2 * TP + FP + FN),
        'IoU':       safe(TP, TP + FP + FN),
    }

# Load best checkpoint (prefer Phase 2)
ckpt_path = os.path.join(LOG_DIR, '01', 'net_best_checkpoint.tar')
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(LOG_DIR, '00', 'net_best_checkpoint.tar')
print(f'Checkpoint: {ckpt_path}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = create_model(input_ch=3, nr_types=NR_TYPES, freeze=False, mode=MODEL_MODE)
model = torch.nn.DataParallel(model).to(device)
checkpoint = torch.load(ckpt_path)
model.load_state_dict(checkpoint['desc'])
model.eval()

# Setup test loader
test_file_list = sorted(glob.glob(os.path.join(TEST_PATCH_DIR, '*.npy')))
if MODEL_MODE == 'original':
    act_shape, out_shape = [270, 270], [80, 80]
else:
    act_shape, out_shape = [256, 256], [164, 164]

original_init = FileLoader.__init__
def patched_init(self, *args, **kwargs):
    original_init(self, *args, **kwargs)
    if not hasattr(self, 'shape_augs'): self.shape_augs = None
    if not hasattr(self, 'input_augs'): self.input_augs = None
FileLoader.__init__ = patched_init

test_dataset = FileLoader(
    test_file_list, mode='valid', with_type=TYPE_CLASSIFICATION,
    setup_augmentor=False, input_shape=act_shape, mask_shape=out_shape,
    target_gen=(gen_targets, {})
)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

if os.path.exists(PRED_SAVE_DIR):
    shutil.rmtree(PRED_SAVE_DIR)
os.makedirs(PRED_SAVE_DIR)

metrics_acc = {k: [] for k in ['Accuracy', 'Precision', 'Recall', 'Dice', 'IoU']}

print(f'Running inference on {len(test_file_list)} test patches...')
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm.tqdm(test_loader, desc='Testing')):
        # FIX: Model expects [0, 255] input range, removed division by 255.0
        imgs = batch['img'].permute(0, 3, 1, 2).float().to(device)
        true_np = batch['np_map'].numpy()
        output = model(imgs)

        # Get NP predictions — use raw logits (no double-softmax)
        pred_logits = output['np']
        pred_np_cls = torch.argmax(pred_logits, dim=1).cpu().numpy()

        # Crop GT and pred to same spatial size
        ph, pw = pred_np_cls.shape[1], pred_np_cls.shape[2]
        gh, gw = true_np.shape[1], true_np.shape[2] if true_np.ndim == 3 else (true_np.shape[1],)
        if true_np.ndim == 3:
            gh, gw = true_np.shape[1], true_np.shape[2]
        else:
            gh, gw = true_np.shape[1], true_np.shape[1]
        min_h, min_w = min(ph, gh), min(pw, gw)
        pred_crop = pred_np_cls[:, :min_h, :min_w]
        true_crop = true_np[:, :min_h, :min_w]

        for i in range(len(true_crop)):
            gt_mask   = (true_crop[i] > 0).astype(np.uint8)
            pred_mask = pred_crop[i].astype(np.uint8)
            m = calculate_metrics(gt_mask, pred_mask)
            for k, v in m.items():
                metrics_acc[k].append(v)
            global_idx = batch_idx * test_loader.batch_size + i
            if global_idx < len(test_file_list):
                fname = os.path.basename(test_file_list[global_idx]).replace('.npy', '.png')
                cv2.imwrite(os.path.join(PRED_SAVE_DIR, fname), pred_mask * 255)

print(f'\n{"="*45}')
print('  TEST SET RESULTS')
print(f'{"="*45}')
for k, v in metrics_acc.items():
    print(f'  {k:<12}: {np.mean(v):.4f}')
print(f'{"="*45}')

# Show a few predictions
sample_files = sorted(glob.glob(os.path.join(TEST_PATCH_DIR, '*.npy')))[:3]
pred_files   = sorted(glob.glob(os.path.join(PRED_SAVE_DIR, '*.png')))[:3]
if sample_files and pred_files:
    fig, axes = plt.subplots(len(sample_files), 3, figsize=(12, 4 * len(sample_files)))
    if len(sample_files) == 1:
        axes = [axes]
    for i in range(len(sample_files)):
        data = np.load(sample_files[i])
        img = data[..., :3].astype('uint8')
        gt  = (data[..., 3] > 0).astype('uint8')
        pred = cv2.imread(pred_files[i], cv2.IMREAD_GRAYSCALE)
        pred = (pred > 0).astype('uint8')
        # Center crop GT and image to match prediction size
        ch, cw = pred.shape[:2], pred.shape[:2]
        axes[i][0].imshow(img); axes[i][0].set_title('Image'); axes[i][0].axis('off')
        axes[i][1].imshow(gt, cmap='gray'); axes[i][1].set_title('Ground Truth'); axes[i][1].axis('off')
        axes[i][2].imshow(pred, cmap='gray'); axes[i][2].set_title('Prediction'); axes[i][2].axis('off')
    plt.tight_layout(); plt.show()

Checkpoint: /content/hovernet_logs/01/net_best_checkpoint.tar
Running inference on 539 test patches...



Testing: 100%|██████████| 68/68 [00:22<00:00,  3.02it/s]



  TEST SET RESULTS
  Accuracy    : 0.9290
  Precision   : 0.8168
  Recall      : 0.8521
  Dice        : 0.8273
  IoU         : 0.7153
